# SAC Irrigation Training - v2.11 (LayerNorm critic, Kaggle)

**Algorithm:** SAC (stable_baselines3) with VDN-factorised twin-Q critic + LayerNorm hidden layers
**Replay buffer:** SB3 standard 1-step `ReplayBuffer` (NOT the E3 NStepReplayBuffer)
**Single change vs v2.7:** LayerNorm inserted after each hidden Linear in the critic
**Gamma:** 0.99 (v2.7 baseline, UNCHANGED)

## Why v2.11 exists

Phase 1 (v2.10 E2/E3/E4) ruled out three approaches to the v2.7 deadly-triad cascade. v2.11 tests the hypothesis that the cascade is driven by neural-network optimization dynamics (Yue NeurIPS 2023, Nauman RLC 2024): inserting LayerNorm after each hidden Linear in the critic suppresses the Self-Excite Eigenvalue Measure (SEEM) and prevents Q-divergence.

The v2.7 critic_loss trajectory grows roughly factor-of-10 per 10k steps from step 150k onward - clean exponential, the signature LayerNorm is designed to suppress.

## Acceptance criterion

Primary (cascade suppressed):
- `|q_inflation_pct| < 30%` throughout 250k steps
- `critic_loss` never exceeds ~50 past step 100k
- `actor/std/spatial` stays in [0.20, 0.40] throughout

Secondary (policy quality preserved):
- 9-cell yields within +-3% of v2.7 best_model (seed 0)

## Early-kill rule

If at any checkpoint past step 150k BOTH `q_inflation_pct > 100%` AND `actor/std/spatial < 0.15`, stop the run.

## Kaggle environment

- GPU: T4 x 1 (Accelerator: GPU T4 x2 reduces to T4 x 1 for SB3)
- ~2-2.5 hours per 250k-step run
- WandB API key must be added under **Settings -> Secrets** as `WANDB_API_KEY`


In [ ]:
# Cell 1: Clone repo and install deps.
import subprocess, sys, os

WORK_ROOT = '/kaggle/working'
REPO_DIR  = f'{WORK_ROOT}/thesis'

if os.path.exists(REPO_DIR):
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', REPO_DIR],
    check=True
)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install SB3 - sb3-contrib NOT required for v2.11 (no TQC).
subprocess.run(
    ['pip', 'install', '--quiet',
     'stable-baselines3==2.6.0',
     'gymnasium', 'wandb', 'pytest'],
    check=True
)

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 2: WandB secret + GPU check.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('OK  WANDB_API_KEY loaded from Kaggle Secrets.')
except Exception as e:
    print(f'NOTE: Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('     Training continues without WandB - add WANDB_API_KEY under Settings -> Secrets to enable it.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')


In [ ]:
# Cell 3: Pre-training validation.
#
# Runs smoke tests, factorized-critic tests (which include v2.11 LayerNorm
# critic shape and param-count guards), and a 1000-step pilot to catch
# import/wiring bugs.  Abort if anything fails.

import subprocess, sys

print('Smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nFactorized-critic tests (v2.7 + v2.11 architectures)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'FACTORIZED CRITIC TESTS FAILED'

print('\n1000-step pilot training (wiring check, ~1 minute)...')
from src.rl.train_v211 import train_sac_v211
_ = train_sac_v211(
    seed=999,
    output_dir='/kaggle/working/pilot',
    wandb_project=None,
    total_timesteps=1000,
)
print('\nOK  Pre-flight passed. Proceed to Cell 4.')


In [ ]:
# Cell 4: Full 250k training (SAC v2.11, LayerNorm critic, gamma=0.99).
# Kaggle T4: ~2-2.5 hours per 250k-step run.
#
# Start with SEED=0 (paired with v2.7 seed 0 published numbers).
# Expand to seeds 1, 2 only after seed-0 results meet the acceptance criterion.

SEED = 0       # CHANGE per session

from src.rl.train_v211 import train_sac_v211

model = train_sac_v211(
    seed=SEED,
    output_dir='/kaggle/working/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    gamma=0.99,      # v2.7 baseline; LayerNorm is the experimental variable
)
print('Training complete.')


In [ ]:
# Cell 5: Archive results so Kaggle persists them after the session.
#
# Kaggle keeps /kaggle/working in 'Output' between sessions for the SAME
# notebook version, but a fresh fork or session can lose them.  Compress
# the run directory so it shows up cleanly as a downloadable artifact.
import shutil, os

src = f'/kaggle/working/results/rl/sac_v211_seed{SEED}'

# Remove the replay buffer before archiving (saves several hundred MB).
rb = os.path.join(src, 'replay_buffer_latest.pkl')
if os.path.exists(rb):
    os.remove(rb)
    print(f'Removed {rb} for compact archive.')

archive_base = f'/kaggle/working/sac_v211_seed{SEED}'
shutil.make_archive(archive_base, 'zip', src)
size_mb = os.path.getsize(archive_base + '.zip') / 1024 / 1024
print(f'Archive: {archive_base}.zip  ({size_mb:.1f} MB)')

# List what's inside the archive.
import zipfile
with zipfile.ZipFile(archive_base + '.zip') as zf:
    for name in zf.namelist()[:30]:
        info = zf.getinfo(name)
        print(f'  {name}  ({info.file_size/1024:.1f} KB)')
    if len(zf.namelist()) > 30:
        print(f'  ... and {len(zf.namelist())-30} more files')


In [ ]:
# Cell 6: Post-training 9-cell evaluation (SAC eval path).
#
# v2.11 produces a SAC checkpoint, so use exp_rl.py (not exp_rl_tqc.py).
# The runner auto-detects the LayerNorm critic via the 1-D 'critic.qf0.1.weight'
# key and dispatches to V211CTDESACPolicy.  The observation builder uses the
# v2.7 path (8 features/agent, 1097-dim) since v2.11's actor and obs layout
# are unchanged from v2.7.
import subprocess, sys

model_path = f'/kaggle/working/results/rl/sac_v211_seed{SEED}/best_model/best_model.zip'

print('Evaluating on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',     'eval',
    '--model',    model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'perfect',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL FAILED'

print('\nEvaluating on 9-cell grid (noisy forecast, seed=42)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode',       'eval',
    '--model',      model_path,
    '--scenario',   'all',
    '--budget',     'all',
    '--forecast',   'noisy',
    '--noise-seed', '42',
], capture_output=False)
if r.returncode != 0:
    print('Noisy-forecast eval failed; perfect-forecast only.')


In [ ]:
# Cell 7: Q-inflation trajectory plot (v2.11 cascade diagnostic).
#
# With gamma=0.99 (UNCHANGED from v2.7), the structural baseline is the same:
#     Q_structural = alpha * geom_weight * (-log pi_start)
#     geom_weight = min(1/(1-gamma), 93) = 93
#     With alpha=0.05 and -log pi_start ~ 89:  Q_structural ~ 414
# v2.7 measured ~378-414 BEFORE the cascade then exploded to >+200% post 175k.
# v2.11 should stay <+30% throughout if LayerNorm has done its job.

import pandas as pd
import matplotlib.pyplot as plt

csv_path = f'/kaggle/working/results/rl/sac_v211_seed{SEED}/bias_ratio_log.csv'
df = pd.read_csv(csv_path)
print(df.tail(10))

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax = axes[0]
ax.plot(df['step'], df['q_pred_mean'],  '-o', label='Q_pred_mean',                  color='C0')
ax.plot(df['step'], df['q_structural'], '-s', label='Q_structural (theoretical)',   color='C1')
ax.axhline(0, color='k', linestyle=':', alpha=0.3)
ax.set_ylabel('Q value')
ax.set_title(f'v2.11 (SAC, LayerNorm critic, gamma=0.99) seed {SEED} - Q calibration')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(df['step'], df['q_inflation_pct'], '-o', color='C3', label='q_inflation_pct')
ax.axhline(0.0,   color='k',      linestyle='-',  alpha=0.4, label='ideal')
ax.axhline(30.0,  color='g',      linestyle=':',  alpha=0.6, label='acceptance threshold (30%)')
ax.axhline(-30.0, color='g',      linestyle=':',  alpha=0.6)
ax.axhline(50.0,  color='orange', linestyle=':',  alpha=0.6, label='cascade onset (50%)')
ax.axhline(-50.0, color='orange', linestyle=':',  alpha=0.6)
ax.axhline(200.0, color='r',      linestyle=':',  alpha=0.6, label='full cascade (200%)')
ax.axhline(-200.0,color='r',      linestyle=':',  alpha=0.6)
ax.set_ylabel('Q_inflation %')
ax.set_xlabel('training step')
ax.legend(loc='best', fontsize='small')
ax.grid(alpha=0.3)

plt.tight_layout()
plot_path = f'/kaggle/working/results/rl/sac_v211_seed{SEED}/q_inflation_trajectory.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved plot: {plot_path}')


In [ ]:
# Cell 8: Resume from checkpoint (if session was interrupted).
# Upload the previous run as a Kaggle Dataset, attach it to this notebook,
# then uncomment and fill in CHECKPOINT_STEP / DATASET_PATH.

# SEED  = 0
# CHECKPOINT_STEP = 100_000
# DATASET_PATH = f'/kaggle/input/sac-v211-seed{SEED}'   # Dataset slug after attaching
#
# import shutil, os
# local_dir = f'/kaggle/working/results/rl/sac_v211_seed{SEED}'
# os.makedirs(local_dir, exist_ok=True)
# shutil.copytree(DATASET_PATH, local_dir, dirs_exist_ok=True)
#
# from stable_baselines3 import SAC
# from src.rl.networks import V211CTDESACPolicy
# ckpt = f'{local_dir}/checkpoints/sac_v211_seed{SEED}_{CHECKPOINT_STEP}_steps.zip'
# model = SAC.load(ckpt, custom_objects={'policy_class': V211CTDESACPolicy})
# # Continue training with model.learn(total_timesteps=..., reset_num_timesteps=False)
